# TFMv3 en Google Colab con GPU

Notebook reproducible para repetir el experimento principal del autoencoder en Google Colab usando GPU. La primera parte comprueba el entorno y ejecuta una prueba reducida. La segunda parte lanza la campana cientifica completa del AE con semillas 13, 42 y 73 y compara el AUROC con el resultado local de referencia: **0.7608**.

Antes de ejecutar, selecciona `Runtime > Change runtime type > GPU`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import time

REPO_URL = "https://github.com/alerodriargui/TFMv3.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/TFMv3")
LOCAL_AUROC_REFERENCE = 0.7608

print("Python", sys.version)
print("Platform", platform.platform())

## 1. Clonar el repositorio e instalar dependencias

Colab ya incluye PyTorch compatible con su runtime CUDA. Por eso se instalan solo las dependencias de apoyo desde `requirements-colab.txt` y despues se comprueba la version real de PyTorch/CUDA.

In [ ]:
if PROJECT_DIR.exists():
    print(f"Repositorio ya presente en {PROJECT_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Trabajo en", Path.cwd())
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

In [ ]:
%pip install -q -r requirements-colab.txt

## 2. Comprobar GPU, PyTorch y CUDA

Esta celda falla de forma explicita si la sesion no tiene CUDA. Si ocurre, cambia el tipo de runtime a GPU y vuelve a ejecutar desde el principio.

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", cuda_available)
print("CUDA PyTorch:", torch.version.cuda)

if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except FileNotFoundError:
        print("nvidia-smi no esta disponible en esta runtime")
else:
    raise RuntimeError("Esta ejecucion necesita GPU. Activa Runtime > Change runtime type > GPU.")

## 3. Montar Google Drive y localizar Chest-RSNA

No subas datos medicos ni credenciales al repositorio. Coloca `Chest-RSNA` en Drive, por ejemplo en `/content/drive/MyDrive/datasets/Chest-RSNA`, manteniendo esta estructura:

```text
Chest-RSNA/
  train/good/
  valid/good/
  valid/Ungood/
  test/good/
  test/Ungood/
```

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/datasets/Chest-RSNA")
os.environ["TFM_DATA_ROOT"] = str(DATA_ROOT)
print("TFM_DATA_ROOT=", os.environ["TFM_DATA_ROOT"])

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"No existe {DATA_ROOT}. Ajusta DATA_ROOT a la ubicacion real de Chest-RSNA en Drive.")

In [ ]:
from data import find_images, resolve_data_root, split_dir

root = resolve_data_root(DATA_ROOT)
expected = {
    "train/good": split_dir(root, "train") / "good",
    "valid/good": split_dir(root, "val") / "good",
    "valid/Ungood": split_dir(root, "val") / "Ungood",
    "test/good": split_dir(root, "test") / "good",
    "test/Ungood": split_dir(root, "test") / "Ungood",
}

dataset_counts = {}
for name, folder in expected.items():
    if not folder.is_dir():
        raise FileNotFoundError(f"Falta la carpeta {folder}")
    dataset_counts[name] = len(find_images(folder))

print("Dataset:", root)
print(json.dumps(dataset_counts, indent=2, ensure_ascii=False))

## 4. Prueba reducida en GPU

Esta prueba usa pocas imagenes para confirmar que el pipeline completo entrena, calibra, evalua y escribe artefactos sobre GPU antes de lanzar la campana completa.

In [ ]:
SMOKE_OUTPUT = Path("artifacts/colab_smoke")
smoke_command = [
    sys.executable,
    "run.py",
    "--seeds",
    "13",
    "--epochs",
    "1",
    "--batch-size",
    "32",
    "--max-train-images",
    "128",
    "--max-eval-images-per-class",
    "32",
    "--output-root",
    str(SMOKE_OUTPUT),
    "--data-root",
    str(root),
]

print("Comando:", " ".join(smoke_command))
started = time.perf_counter()
subprocess.run(smoke_command, check=True)
print(f"Duracion prueba reducida: {time.perf_counter() - started:.1f} s")

In [ ]:
smoke_metrics_path = SMOKE_OUTPUT / "ae_seed13" / "metrics.json"
smoke_metrics = json.loads(smoke_metrics_path.read_text(encoding="utf-8"))
print(json.dumps({
    "device": smoke_metrics["device"],
    "elapsed_seconds": smoke_metrics["elapsed_seconds"],
    "test_auroc": smoke_metrics["test"]["auroc"],
    "test_balanced_accuracy": smoke_metrics["test"]["balanced_accuracy"],
    "scientific_run": smoke_metrics["scientific_run"],
}, indent=2))

assert smoke_metrics["device"] == "cuda", smoke_metrics["device"]

## 5. Campana completa del AE final

Ejecuta el AE final con semillas 13, 42 y 73, 3 epocas, batch 32 y test completo. Esta es la ejecucion comparable con el resultado local documentado.

In [ ]:
FULL_OUTPUT = Path("artifacts/colab_ae_full")
full_command = [
    sys.executable,
    "run.py",
    "--seeds",
    "13",
    "42",
    "73",
    "--epochs",
    "3",
    "--batch-size",
    "32",
    "--output-root",
    str(FULL_OUTPUT),
    "--data-root",
    str(root),
]

print("Comando:", " ".join(full_command))
started = time.perf_counter()
subprocess.run(full_command, check=True)
full_elapsed = time.perf_counter() - started
print(f"Duracion campana AE completa: {full_elapsed:.1f} s")

In [ ]:
import pandas as pd

rows = []
for metrics_path in sorted(FULL_OUTPUT.glob("ae_seed*/metrics.json")):
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    rows.append({
        "seed": metrics["config"]["seed"],
        "device": metrics["device"],
        "selected_epoch": metrics["selected_epoch"],
        "elapsed_seconds": metrics["elapsed_seconds"],
        "auroc": metrics["test"]["auroc"],
        "balanced_accuracy": metrics["test"]["balanced_accuracy"],
        "scientific_run": metrics["scientific_run"],
    })

summary = pd.DataFrame(rows).sort_values("seed")
display(summary)

mean_auroc = float(summary["auroc"].mean())
print(f"AUROC medio Colab GPU: {mean_auroc:.4f}")
print(f"AUROC local referencia: {LOCAL_AUROC_REFERENCE:.4f}")
print(f"Diferencia: {mean_auroc - LOCAL_AUROC_REFERENCE:+.4f}")

assert set(summary["seed"]) == {13, 42, 73}
assert (summary["device"] == "cuda").all()
assert summary["scientific_run"].all()

## 6. Guardar artefactos en Drive

Se guardan resultados, metricas, tiempos, reconstrucciones y checkpoints. El archivo `modelo_autoencoder.pt` corresponde a la semilla 42, igual que en la ejecucion local.

In [ ]:
from datetime import datetime
import shutil

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_output = Path("/content/drive/MyDrive/TFMv3_colab_outputs") / timestamp
drive_output.mkdir(parents=True, exist_ok=True)

for path in [Path("resultados.csv"), Path("modelo_autoencoder.pt")]:
    if path.exists():
        shutil.copy2(path, drive_output / path.name)

shutil.copytree(FULL_OUTPUT, drive_output / "artifacts_colab_ae_full", dirs_exist_ok=True)
shutil.copytree(SMOKE_OUTPUT, drive_output / "artifacts_colab_smoke", dirs_exist_ok=True)

run_manifest = {
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "data_root": str(root),
    "dataset_counts": dataset_counts,
    "smoke_command": smoke_command,
    "full_command": full_command,
    "full_elapsed_seconds": full_elapsed,
    "local_auroc_reference": LOCAL_AUROC_REFERENCE,
    "colab_auroc_mean": mean_auroc,
    "colab_minus_local_auroc": mean_auroc - LOCAL_AUROC_REFERENCE,
}
(drive_output / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("Artefactos guardados en", drive_output)

## Interpretacion de diferencias CPU local vs GPU Colab

Si los AUROC difieren ligeramente de 0.7608, revisa primero el manifiesto: GPU asignada, versiones de PyTorch/CUDA, conteos del dataset y commit. El entrenamiento fija semillas y configura cuDNN en modo determinista, pero pequenas diferencias numericas pueden aparecer entre CPU y GPU o entre versiones de PyTorch/CUDA. Una diferencia relevante debe documentarse junto con esos datos y con los `metrics.json` de cada semilla.